In [18]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

csv_path_brandvsfree = Path(r"D:\Tankdaten\rq_brand_vs_free\brand_vs_free_yearly_stats.csv")
csv_per_brand = Path(r"D:\Tankdaten\rq_brand_vs_free\per_brand_yearly_stats.csv")

In [22]:
df_brandvsfree = pl.read_csv(csv_path_brandvsfree).filter(pl.col("year") < 2026) # Filter out 2026 data because it's incomplete
df_per_brand = pl.read_csv(csv_per_brand).filter(pl.col("year") < 2026) # Filter out 2026 data because it's incomplete

brand = df_brandvsfree.filter(pl.col("station_type") == "brand_station").sort("year")
free  = df_brandvsfree.filter(pl.col("station_type") == "free_station").sort("year")
aral = df_per_brand.filter(pl.col("brand_normalized") == "ARAL").sort("year")

print(df_brandvsfree)
print(df_per_brand)

shape: (24, 6)
┌───────────────┬──────┬────────────────────┬─────────────┬──────────┬──────────┐
│ station_type  ┆ year ┆ price_update_count ┆ diesel_mean ┆ e5_mean  ┆ e10_mean │
│ ---           ┆ ---  ┆ ---                ┆ ---         ┆ ---      ┆ ---      │
│ str           ┆ i64  ┆ i64                ┆ f64         ┆ f64      ┆ f64      │
╞═══════════════╪══════╪════════════════════╪═════════════╪══════════╪══════════╡
│ brand_station ┆ 2014 ┆ 9073214            ┆ 1.339087    ┆ 1.526732 ┆ 1.486627 │
│ brand_station ┆ 2015 ┆ 20197781           ┆ 1.173514    ┆ 1.395535 ┆ 1.375351 │
│ brand_station ┆ 2016 ┆ 25553204           ┆ 1.091302    ┆ 1.311333 ┆ 1.290864 │
│ brand_station ┆ 2017 ┆ 34236480           ┆ 1.168615    ┆ 1.374559 ┆ 1.350754 │
│ brand_station ┆ 2018 ┆ 38620946           ┆ 1.293682    ┆ 1.459982 ┆ 1.436029 │
│ …             ┆ …    ┆ …                  ┆ …           ┆ …        ┆ …        │
│ free_station  ┆ 2021 ┆ 44639144           ┆ 1.378195    ┆ 1.571538 ┆ 1.514523 │
│

## Mean Fuel Prices: Brand vs. Free Stations (per year)

In [20]:
fuels = [("diesel_mean", "Diesel"), ("e5_mean", "E5"), ("e10_mean", "E10")]

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[label for _, label in fuels])

# Claude Code Start
for col_i, (col, label) in enumerate(fuels, start=1):
    show_legend = col_i == 1
    fig.add_trace(go.Scatter(
        x=brand["year"], 
        y=brand[col].round(4),
        mode="lines+markers", 
        name="Brand station",
        line=dict(color="steelblue", width=2),
        legendgroup="brand", 
        showlegend=show_legend,
        hovertemplate="%{x}: %{y:.3f} €/L",
    ), row=1, col=col_i)
    # Claude Code Ende
    fig.add_trace(go.Scatter(
        x=free["year"], 
        y=free[col].round(4),
        mode="lines+markers", 
        name="Free station",
        line=dict(color="darkorange", width=2, dash="dash"),
        legendgroup="free", 
        showlegend=show_legend,
        hovertemplate="%{x}: %{y:.3f} €/L", # Claude
    ), row=1, col=col_i)

fig.update_yaxes(ticksuffix=" €", col=1)
fig.update_layout(
    title="Mean Fuel Prices: Brand vs. Free Stations (Germany)",
    height=450, 
    width=1000,
    hovermode="x unified",
)
fig.show()

## Price Difference: Brand minus Free (per year)

In [ ]:
years = brand["year"].to_list()

colors = {"Diesel": "steelblue", "E5": "darkorange", "E10": "seagreen"}

fig = go.Figure()
fig.add_hline(y=0, line_color="black", line_width=1)

for col, label in fuels:
    diff_ct = [(b - f) * 100 for b, f in zip(brand[col].to_list(), free[col].to_list())]
    fig.add_trace(go.Scatter(
        x=years, 
        y=[round(v, 3) for v in diff_ct],
        mode="lines+markers", 
        name=label,
        line=dict(color=colors[label], width=2),
        hovertemplate="%{x}: %{y:+.2f} ct/L",
    ))
    avg = sum(diff_ct) / len(diff_ct)
    print(f"{label}: avg difference = {avg:+.2f} ct/L")

fig.update_layout(
    title="Price Difference: Brand − Free (positive = brand is more expensive)",
    xaxis_title="Year",
    yaxis_title="Difference (ct/L)",
    yaxis_ticksuffix=" ct",
    hovermode="x unified",
    height=450, width=900,
)
fig.show()

Diesel: avg difference = +2.01 ct/L
E5: avg difference = +1.86 ct/L
E10: avg difference = +1.74 ct/L


## Price Difference: ARAL - other